# 01 — Public matches dataset and the first training slice

Explores BAR's public match dumps from
[`data-processing`](https://github.com/beyond-all-reason/data-processing) and picks the games
Reason8 learns from first. Summary in [`docs/reference/dataset.md`](../docs/reference/dataset.md).

The Parquet dumps are downloaded once into the git-ignored data dir (`data/matches/`, or
`$REASON8_DATA_DIR/matches/`). **Privacy:** player names are personal data, so this notebook
never prints them; it only shows ids and aggregates.

In [1]:
from datetime import datetime

import duckdb
import httpx
import polars as pl

from reason8.data import matches as dumps

pl.Config.set_tbl_rows(12)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_tbl_hide_dataframe_shape(True)

paths = dumps.download_dumps()
for name, path in paths.items():
    print(f"{name:24s} {path.stat().st_size / 1e6:7.1f} MB")

matches                     56.6 MB
match_players              290.9 MB
players                      1.5 MB
player_skill_snapshot        4.3 MB


In [2]:
matches = pl.read_parquet(paths["matches"])
match_players = dumps.scan_dump("match_players")  # 16M rows: keep it lazy
db = duckdb.connect()  # DuckDB reads the Parquet file directly
db.sql(f"create view matches as select * from read_parquet('{paths['matches']}')")

print("matches:", matches.height, "rows")
print("match_players:", match_players.select(pl.len()).collect().item(), "rows")
print("start_time:", matches["start_time"].min(), "->", matches["start_time"].max())
matches.schema

matches: 1583687 rows
match_players: 16390230 rows
start_time: 2022-06-02 02:20:03 -> 2026-11-21 15:49:39


Schema([('match_id', Int64),
        ('replay_id', String),
        ('replay_file_name', String),
        ('start_time', Datetime(time_unit='us', time_zone=None)),
        ('map', String),
        ('team_count', Int32),
        ('game_type', String),
        ('winning_team', Int32),
        ('game_duration', Int32),
        ('is_ranked', Boolean),
        ('engine', String),
        ('game_version', String),
        ('is_public', Boolean),
        ('has_deleted_player', Boolean)])

In [3]:
match_players.collect_schema()

Schema([('match_id', Int64),
        ('team_id', Int32),
        ('user_id', Int64),
        ('party_id', String),
        ('left_after', Int32),
        ('old_skill', Float32),
        ('old_uncertainty', Float32),
        ('new_skill', Float32),
        ('new_uncertainty', Float32),
        ('faction', String)])

Beyond the obvious columns, `matches` has `team_count`, **`engine`**
(the Recoil engine version, e.g. `2026.09.03`), `is_public` and `has_deleted_player`;
`match_players` has `party_id`, `old/new_skill`, `old/new_uncertainty` (OpenSkill mu and sigma)
and `left_after` (seconds into the game when the player left).

## Counts by game type, month, map, faction

In [4]:
matches.group_by("game_type").agg(
    pl.len().alias("matches"),
    pl.col("is_ranked").sum().alias("ranked"),
    pl.col("start_time").min().dt.date().alias("first"),
).sort("matches", descending=True)

game_type,matches,ranked,first
str,u32,u32,date
"""Large Team""",614912,599769,2024-06-11
"""Duel""",301754,257938,2022-06-02
"""Team""",258368,258368,2022-06-02
"""Small Team""",216670,181850,2024-06-11
"""Bots""",124209,0,2025-11-21
"""FFA""",40738,35986,2022-06-02
"""Team FFA""",12481,6328,2022-06-03
"""Raptors""",10023,0,2025-11-21
"""Scavengers""",4532,0,2025-11-21


In [5]:
# matches per quarter and game type, in DuckDB
db.sql("""
    pivot (
        select date_trunc('quarter', start_time)::date as quarter, game_type from matches
        where game_type in ('Large Team', 'Team', 'Small Team', 'Duel')
    )
    on game_type using count(*) group by quarter order by quarter
""").show(max_rows=30)

┌────────────┬───────┬────────────┬────────────┬───────┐
│  quarter   │ Duel  │ Large Team │ Small Team │ Team  │
│    date    │ int64 │   int64    │   int64    │ int64 │
├────────────┼───────┼────────────┼────────────┼───────┤
│ 2022-04-01 │   864 │          0 │          0 │   897 │
│ 2022-07-01 │  2427 │          0 │          0 │  3976 │
│ 2022-10-01 │  3300 │          0 │          0 │  4776 │
│ 2023-01-01 │  7668 │          0 │          0 │ 11529 │
│ 2023-04-01 │ 16120 │          0 │          0 │ 57488 │
│ 2023-07-01 │ 11706 │          0 │          0 │ 45236 │
│ 2023-10-01 │  8953 │          0 │          0 │ 40805 │
│ 2024-01-01 │ 14853 │          0 │          0 │ 53742 │
│ 2024-04-01 │ 15044 │       7868 │       2351 │ 39919 │
│ 2024-07-01 │ 13121 │      36083 │       9238 │     0 │
│ 2024-10-01 │ 14954 │      35781 │      11787 │     0 │
│ 2025-01-01 │ 15739 │      43152 │      14448 │     0 │
│ 2025-04-01 │ 21559 │      78247 │      22658 │     0 │
│ 2025-07-01 │ 19093 │      844

"Team" (ranked) stops in early 2024 when BAR split team games into **Small Team** and
**Large Team**. Large Team (6v6–8v8) is now the biggest queue by far.

In [6]:
# players per match, by game type (last 6 months)
recent = matches.filter(pl.col("start_time") >= datetime(2026, 5, 23))
sizes = dumps.lobby_stats(match_players.join(recent.lazy().select("match_id"), on="match_id"))
(
    recent.join(sizes.collect(), on="match_id")
    .group_by("game_type", "n_players")
    .len()
    .sort("game_type", "len", descending=[False, True])
    .group_by("game_type", maintain_order=True)
    .head(2)
)

game_type,n_players,len
str,u32,u32
"""Bots""",8,20488
"""Bots""",2,13449
"""Duel""",2,61468
"""FFA""",3,3005
"""FFA""",4,2498
"""Large Team""",16,122882
…,…,…
"""Scavengers""",2,419
"""Scavengers""",8,289


In [7]:
# most played maps, ranked Large Team, last 6 months
(
    recent.filter(pl.col("game_type") == "Large Team", pl.col("is_ranked"))
    .group_by("map")
    .len()
    .with_columns(share=pl.col("len") / pl.col("len").sum())
    .sort("len", descending=True)
    .head(8)
)

map,len,share
str,u32,f64
"""Supreme Isthmus v2.1""",49048,0.414428
"""All That Glitters v2.2.3""",42923,0.362675
"""Rosetta 1.4.4""",2560,0.021631
"""Flats and Forests v2.2""",787,0.00665
"""Bismuth Valley v2.4.1""",770,0.006506
"""The Tartar Steppe v1.1""",744,0.006286
"""Forge v2.3""",623,0.005264
"""Starwatcher 1.0""",595,0.005027


Two maps dominate Large Team: **Supreme Isthmus v2.1** and **All That Glitters v2.2.3** make up
~78% of ranked Large Team games; the long tail has under 3k games per map in six months.

In [8]:
# faction picks per player, by year
(
    match_players.join(matches.lazy().select("match_id", "start_time"), on="match_id")
    .group_by(year=pl.col("start_time").dt.year(), faction=pl.col("faction").fill_null("(null)"))
    .len()
    .collect()
    .pivot("faction", index="year", values="len")
    .sort("year")
)

year,Armada,Cortex,(null),Unknown,Random,Legion
i32,u32,u32,u32,u32,u32,u32
2022,45837,42649,30192,416,45,null
2023,1328015,884539,83322,8494,108,null
2024,1571152,1262205,506,33830,15,null
2025,2692236,2303851,2567,269661,57507,43536
2026,2640124,2362866,18495,113882,97887,496293


## Skill distribution

In [9]:
# old_skill (OpenSkill mu before the match) per player-game, last 6 months
(
    match_players.join(recent.lazy().select("match_id", "game_type"), on="match_id")
    .group_by("game_type")
    .agg(
        pl.len().alias("player_games"),
        *[
            pl.col("old_skill").quantile(q).round(1).alias(f"p{int(q * 100)}")
            for q in (0.1, 0.25, 0.5, 0.75, 0.9, 0.99)
        ],
        pl.col("old_uncertainty").median().round(2).alias("sigma_p50"),
    )
    .sort("player_games", descending=True)
    .collect()
)

game_type,player_games,p10,p25,p50,p75,p90,p99,sigma_p50
str,u32,f32,f32,f32,f32,f32,f32,f32
"""Large Team""",2016888,14.0,17.799999,22.9,29.200001,37.599998,55.0,3.13
"""Small Team""",354324,16.4,20.5,25.0,29.4,34.900002,48.799999,4.62
"""Bots""",288190,null,null,null,null,null,null,null
"""Duel""",122936,17.4,22.299999,27.1,34.599998,42.799999,57.599998,3.95
"""Team FFA""",65220,24.4,25.0,25.6,28.9,36.200001,72.699997,8.0
"""FFA""",36080,21.5,23.799999,25.0,31.200001,41.0,131.5,6.31
"""Raptors""",27101,null,null,null,null,null,null,null
"""Scavengers""",10642,null,null,null,null,null,null,null


New accounts start at mu=25, sigma=8.33. Lobby averages are much narrower than per-player
skill: matchmaking/balance mixes strong and weak players in big team games.

## Replays: availability and age

In [10]:
newest = matches["start_time"].max()
year_ago = newest.replace(year=newest.year - 1)
matches.select(
    pl.len().alias("matches"),
    pl.col("replay_file_name").is_not_null().sum().alias("with_replay_file"),
    pl.col("replay_file_name").is_not_null().mean().round(4).alias("share"),
    (pl.col("replay_file_name").is_not_null() & (pl.col("start_time") < year_ago))
    .sum()
    .alias("with_replay_older_than_1y"),
)

matches,with_replay_file,share,with_replay_older_than_1y
u32,u32,f64,u32
1583687,1569106,0.9908,874560


In [11]:
# matches without a replay file: mostly the newest ones (replays not yet processed)
(
    matches.filter(pl.col("replay_file_name").is_null())
    .group_by(month=pl.col("start_time").dt.truncate("1mo").dt.date())
    .len()
    .sort("month")
    .tail(4)
)

month,len
date,u32
2026-08-01,87
2026-09-01,82
2026-10-01,98
2026-11-01,449


The replay site warns downloads *may be expired if older than a year*. Probe one random replay
per quarter with a `HEAD` request (18 requests, no download):

In [12]:
sample = (
    matches.filter(pl.col("replay_file_name").is_not_null())
    .sample(20_000, seed=4)
    .group_by(quarter=pl.col("start_time").dt.truncate("1q").dt.date())
    .agg(pl.col("replay_file_name").first())
    .sort("quarter")
)
with httpx.Client(timeout=20) as client:
    status = [client.head(dumps.replay_url(f)).status_code for f in sample["replay_file_name"]]
(
    sample.with_columns(http_status=pl.Series(status))
    .group_by(year=pl.col("quarter").dt.year())
    .agg(pl.len().alias("probed"), (pl.col("http_status") == 200).sum().alias("available"))
    .sort("year")
)

year,probed,available
i32,u32,u32
2022,2,0
2023,4,3
2024,4,4
2025,4,4
2026,4,4


Replays back to 2023 are still downloadable; only the 2022 samples are gone (404). The "older
than a year" warning does not bite for a 6-month slice.

## `is_public`: are private lobbies in the dumps?

In [13]:
matches.group_by("is_public", "is_ranked").len().sort("is_public", "is_ranked")

is_public,is_ranked,len
bool,bool,u32
true,false,243448
true,true,1340239


`is_public` is **true for every row**: private/passworded lobbies (e.g. tournament games) are
not in the dumps at all, so we can't get them from here. That's fine for a first slice (ranked
matchmade-style lobbies are what a bot will face anyway); tournament replays would have to come
from elsewhere.

## Engine and game versions

In [14]:
recent.group_by("engine").agg(
    pl.len().alias("matches"),
    pl.col("start_time").min().dt.date().alias("first"),
    pl.col("start_time").max().dt.date().alias("last"),
    pl.col("game_version").n_unique().alias("game_versions"),
).sort("first")

engine,matches,first,last,game_versions
str,u32,date,date,u32
null,1145,2026-05-23,2026-11-21,1
"""2025.06.19""",8124,2026-05-23,2026-05-26,10
"""2025.06.24""",90568,2026-05-24,2026-07-13,63
"""2025.06.23""",1,2026-05-24,2026-05-24,1
"""2026.06.08""",2,2026-06-24,2026-06-24,1
"""2026.06.10""",3,2026-07-04,2026-07-04,1
…,…,…,…,…
"""2026.07.02""",4,2026-08-02,2026-08-02,1
"""2026.07.04""",175763,2026-08-05,2026-11-16,94


The dumps do contain the **engine** version (also the suffix of `replay_file_name`) and the
exact `game_version` (`Beyond All Reason test-<build>-<sha>`). Engine versions follow each other
with short overlaps (a new engine every 1–3 months), while the game version changes every few
days.

## The first slice

The slice uses **team games**, as implemented by
`reason8.data.matches.select_matches()` defaults:

| filter        | value                                                                                                                 |
| ------------- | --------------------------------------------------------------------------------------------------------------------- |
| maps          | `Supreme Isthmus v2.1`, `All That Glitters v2.2.3`, `Rosetta 1.4.4` (exact versions: layouts change between versions) |
| game type     | `Large Team`, ranked, exactly 8v8 (16 players, 2 teams)                                                               |
| time          | last 182 days of the dump; pinned here to 2026-05-23 .. 2026-11-22                                                    |
| duration      | at least 10 min                                                                                                       |
| early leavers | nobody left in the first 600 s                                                                                        |
| skill         | lobby average `old_skill` ≥ that map's median over the games passing all other filters                                |
| replay        | `replay_file_name` present                                                                                            |

In [15]:
SINCE, UNTIL = datetime(2026, 5, 23), datetime(2026, 11, 22)
slice_args = dict(since=SINCE, until=UNTIL)

# funnel: switch filters on one by one
steps = [
    (
        "3 maps, all games",
        dict(
            game_type=None,
            ranked=None,
            team_size=None,
            min_avg_skill=None,
            early_leave_s=None,
            min_duration_s=None,
            require_replay=False,
        ),
    ),
    (
        "+ ranked Large Team",
        dict(
            team_size=None,
            min_avg_skill=None,
            early_leave_s=None,
            min_duration_s=None,
            require_replay=False,
        ),
    ),
    (
        "+ exactly 8v8",
        dict(min_avg_skill=None, early_leave_s=None, min_duration_s=None, require_replay=False),
    ),
    ("+ replay file", dict(min_avg_skill=None, early_leave_s=None, min_duration_s=None)),
    ("+ >= 10 min", dict(min_avg_skill=None, early_leave_s=None)),
    ("+ no leaver < 600 s", dict(min_avg_skill=None)),
    ("+ lobby avg skill >= median", dict()),
]
funnel = []
for label, kw in steps:
    sel = dumps.select_matches(matches, match_players, **slice_args, **kw)
    funnel.append((label, sel.height))
pl.DataFrame(funnel, schema=["step", "matches"], orient="row")

step,matches
str,i64
"""3 maps, all games""",112960
"""+ ranked Large Team""",94531
"""+ exactly 8v8""",93546
"""+ replay file""",93435
"""+ >= 10 min""",89916
"""+ no leaver < 600 s""",85825
"""+ lobby avg skill >= median""",42913


In [16]:
candidates = dumps.select_matches(matches, match_players, **slice_args, min_avg_skill=None)
chosen = dumps.select_matches(matches, match_players, **slice_args)
print("matches in slice:", chosen.height)
# the skill cutoff is the median lobby average per map
candidates.group_by("map").agg(
    pl.len().alias("candidates"),
    pl.col("avg_skill").median().round(2).alias("median_cutoff"),
).join(chosen.group_by("map").agg(pl.len().alias("in_slice")), on="map").sort(
    "in_slice", descending=True
)

matches in slice: 42913


map,candidates,median_cutoff,in_slice
str,u32,f32,u32
"""Supreme Isthmus v2.1""",44263,22.120001,22132
"""All That Glitters v2.2.3""",39278,21.24,19639
"""Rosetta 1.4.4""",2284,34.259998,1142


### Openings per (map, position)

The dumps have no start positions (those come from replay metadata), but in a full 8v8
every one of the 16 start positions is played once per game. So each slice game gives 16
player-openings, i.e. one per position per game, or two per named role if roles are mirrored
across the two teams.

In [17]:
openings = (
    match_players.join(chosen.lazy().select("match_id"), on="match_id")
    .group_by(pl.col("faction").fill_null("(null)"))
    .len()
    .collect()
    .sort("len", descending=True)
)
n = chosen.height
print(f"player-openings: {openings['len'].sum():,} ({n:,} games x 16)")
per_map = chosen.group_by("map").len().sort("len", descending=True)
for m, g in per_map.iter_rows():
    print(f"{m}: {g:,} per position, {2 * g:,} per mirrored role")
openings.with_columns(share=(pl.col("len") / pl.col("len").sum()).round(3))

player-openings: 686,608 (42,913 games x 16)
Supreme Isthmus v2.1: 22,132 per position, 44,264 per mirrored role
All That Glitters v2.2.3: 19,639 per position, 39,278 per mirrored role
Rosetta 1.4.4: 1,142 per position, 2,284 per mirrored role


faction,len,share
str,u32,f64
"""Armada""",358649,0.522
"""Cortex""",295697,0.431
"""Legion""",21636,0.032
"""Random""",8111,0.012
"""Unknown""",2515,0.004


Is that enough? For a per-position opening model on Isthmus and Glitters it is plenty: ~22k
and ~20k examples per start position, each with dozens of build decisions. Rosetta is much smaller
(~1.1k games after its own median cutoff, which is high: Rosetta lobbies are much stronger than on
the other two maps) but still enough for a frequency baseline, and it tests whether the approach
generalises beyond one map. Legion is rare (~3%). The binding constraint will be
replay download/parsing time, not data volume.

For comparison, the same filters on all maps (top 5):

In [18]:
by_map = dumps.select_matches(matches, match_players, **slice_args, maps=None)
by_map.group_by("map").len().sort("len", descending=True).head(5)

map,len
str,u32
"""Supreme Isthmus v2.1""",22132
"""All That Glitters v2.2.3""",19639
"""Rosetta 1.4.4""",1142
"""Flats and Forests v2.2""",364
"""The Tartar Steppe v1.1""",345


### Skill cutoff vs. the season leaderboard

BAR publishes season leaderboards (top 100 per game type) on
[bar-rts.com/leaderboards](https://bar-rts.com/leaderboards), backed by the JSON API
`https://bar-rts.com/api/leaderboard` (season list) and `https://bar-rts.com/api/leaderboard/<season>`.
Only the top 100 is published, so top 500 is estimated from `player_skill_snapshot`.

In [19]:
lb_path = dumps.matches_dir() / "leaderboard_season3.json"
if not lb_path.exists():
    lb_path.write_bytes(httpx.get("https://bar-rts.com/api/leaderboard/3", timeout=30).content)
lb = pl.read_json(lb_path).explode("players")
large = (
    lb.filter(pl.col("name") == "Large Team")
    # take only id and rating: player names are never loaded into a column
    .select(pl.col("players").struct.field("id"), pl.col("players").struct.field("rating"))
    .with_row_index("rank", offset=1)
)
snapshot = pl.read_parquet(paths["player_skill_snapshot"]).drop("name")
check = large.join(snapshot, on="id", how="left").with_columns(
    k=(pl.col("rating") - pl.col("team_skill")) / pl.col("team_skill_un")
)
print("leaderboard players found in snapshot:", check["team_skill"].is_not_null().sum(), "/ 100")
print("median (rating - mu) / sigma:", round(check["k"].median(), 3), "-> rating = mu - 3 sigma")
print(
    "Large Team #100 rating:",
    round(large["rating"].min(), 2),
    " mu of #100:",
    round(check.sort("rank")["team_skill"][-1], 2),
)

leaderboard players found in snapshot: 100 / 100
median (rating - mu) / sigma: -3.0 -> rating = mu - 3 sigma
Large Team #100 rating: 40.54  mu of #100: 49.05


In [20]:
# top-N in the snapshot by leaderboard rating (mu - 3 sigma), players active in the slice window
active = (
    snapshot.filter(pl.col("last_team") >= SINCE, pl.col("team_skill").is_not_null())
    .with_columns(rating=pl.col("team_skill") - 3 * pl.col("team_skill_un"))
    .sort("rating", descending=True)
    .with_row_index("rank", offset=1)
)
ref = active.filter(pl.col("rank").is_in([100, 500, 1000, 5000])).select(
    "rank", pl.col("rating").round(2), mu=pl.col("team_skill").round(2)
)
print("active Large Team players in window:", active.height)
ref

active Large Team players in window: 30414


rank,rating,mu
u32,f32,f32
100,41.84,49.75
500,30.799999,38.419998
1000,25.469999,34.639999
5000,13.41,25.0


The leaderboard's #100 (rating 40.5) matches the snapshot's #100 among active players (≈41.8);
in `old_skill` (mu) terms **top 100 ≈ mu 48–50**, **top 500 ≈ mu 38**. Lobby averages are far
below that, because even strong lobbies are mixed:

In [21]:
top100_mu, top500_mu = 48.0, 38.4
rows = []
for label, cut in [
    ("median of all candidates", candidates["avg_skill"].median()),
    ("mu 25", 25.0),
    ("mu 30", 30.0),
    ("top 500 (mu 38.4)", top500_mu),
    ("top 100 (mu 48)", top100_mu),
]:
    rows.append((label, round(cut, 1), candidates.filter(pl.col("avg_skill") >= cut).height))
pl.DataFrame(rows, schema=["lobby avg cutoff", "mu", "games"], orient="row")

lobby avg cutoff,mu,games
str,f64,i64
"""median of all candidates""",21.8,42913
"""mu 25""",25.0,26706
"""mu 30""",30.0,13577
"""top 500 (mu 38.4)""",38.4,3625
"""top 100 (mu 48)""",48.0,132


In [22]:
# alternative: keep the median-lobby slice but count openings by top-500/100-level players
strong = (
    match_players.join(chosen.lazy().select("match_id"), on="match_id")
    .select(
        (pl.col("old_skill") >= top500_mu).sum().alias("openings_by_top500_level"),
        (pl.col("old_skill") >= top100_mu).sum().alias("openings_by_top100_level"),
        pl.col("user_id")
        .filter(pl.col("old_skill") >= top500_mu)
        .n_unique()
        .alias("top500_players"),
    )
    .collect()
)
strong

openings_by_top500_level,openings_by_top100_level,top500_players
u32,u32,u32
101228,28536,783


**Decision:** keep the lobby-average cutoff at the **median** per map for the PoC slice (≈43k games over three maps).
Requiring a top-500-level lobby average leaves under 4k games and a top-100 one almost none.
Instead, the leaderboard levels are better used *per player*: the median slice already contains
tens of thousands of openings played by top-500-level players (mu ≥ 38), which can be used to
weight or filter training examples or as a "strong player" evaluation set.

### Engine / game version spread in the slice

Re-simulating a replay likely needs the exact engine + game version. We have engine
`2026.09.03` locally.

In [23]:
chosen.group_by("engine").agg(
    pl.len().alias("games"),
    pl.col("game_version").n_unique().alias("game_versions"),
    pl.col("start_time").min().dt.date().alias("first"),
    pl.col("start_time").max().dt.date().alias("last"),
).with_columns(share=(pl.col("games") / pl.col("games").sum()).round(3)).sort("first")

engine,games,game_versions,first,last,share
str,u32,u32,date,date,f64
"""2025.06.19""",1096,9,2026-05-23,2026-05-26,0.026
"""2025.06.24""",12681,57,2026-05-26,2026-07-13,0.296
"""2026.06.11""",463,4,2026-07-13,2026-07-15,0.011
"""2026.06.12""",5307,40,2026-07-15,2026-08-05,0.124
"""2026.07.04""",22487,89,2026-08-05,2026-11-16,0.524
"""2026.09.02""",65,2,2026-11-16,2026-11-16,0.002
"""2026.09.03""",814,4,2026-11-16,2026-11-21,0.019


In [24]:
print("distinct game_version in slice:", chosen["game_version"].n_unique())
top = chosen.group_by("game_version").len().sort("len", descending=True)
print("games on the 10 most common game versions:", top["len"].head(10).sum())
(
    chosen.filter(pl.col("engine") == "2026.09.03")
    .group_by("game_version")
    .agg(pl.len().alias("games"), pl.col("start_time").min().dt.date().alias("first"))
    .sort("first")
)

distinct game_version in slice: 199
games on the 10 most common game versions: 9826


game_version,games,first
str,u32,date
"""Beyond All Reason test-31986-603f15d""",172,2026-11-16
"""Beyond All Reason test-31992-e7a6cff""",399,2026-11-17
"""Beyond All Reason test-32010-92f8983""",22,2026-11-19
"""Beyond All Reason test-32018-ad135b9""",221,2026-11-20


Only ~2% of the slice (814 games, 4 game versions, since 2026-11-16) was played on engine
`2026.09.03`. Re-simulating the whole slice would need five engines (mostly `2026.07.04`,
`2025.06.24`, `2026.06.12`) and ~200 game versions. The build-order PoC reads commands directly
from the replay and doesn't need re-simulation; the version policy decides how far back to
go, and `select_matches(since=...)` applies it.